# Lesson 6.1 - MLOps Fundamentals & Lifecycle (conceptual demo)

This notebook demonstrates a compact, end-to-end ML lifecycle loop: data load, train/validate, experiment logging, and artifact packaging.

## Objectives

- Build a minimal but reproducible training workflow.
- Log metrics and parameters with MLflow when available.
- Persist model artifacts with version-like metadata.
- Connect notebook steps to CI/CD/CT/CM concepts.

## Setup & Reproducibility

In [ ]:
from __future__ import annotations

import json
import random
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACT_DIR = Path("artifacts/lesson6_1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Data Ingestion and Versioned Snapshot

In [ ]:
raw = load_breast_cancer(as_frame=True)
df = raw.frame.copy()

snapshot_path = ARTIFACT_DIR / "data_snapshot.csv"
df.to_csv(snapshot_path, index=False)

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

{"rows": len(df), "features": X.shape[1], "snapshot": str(snapshot_path)}

## Section B - Training + Evaluation (Experiment Unit)

In [ ]:
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=300, random_state=SEED)),
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
}
metrics

## Section C - MLflow Logging (Optional) + Artifact Packaging

In [ ]:
run_metadata = {
    "run_timestamp": datetime.utcnow().isoformat() + "Z",
    "seed": SEED,
    "model": "logistic_regression",
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "metrics": metrics,
}

model_path = ARTIFACT_DIR / "model.joblib"
metadata_path = ARTIFACT_DIR / "run_metadata.json"

joblib.dump(pipeline, model_path)
metadata_path.write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")

mlflow_status = "not_attempted"
try:
    import mlflow

    mlflow.set_tracking_uri("file:" + str((ARTIFACT_DIR / "mlruns").resolve()))
    mlflow.set_experiment("lesson6_1_mlops_lifecycle")

    with mlflow.start_run(run_name="logreg_baseline"):
        mlflow.log_param("seed", SEED)
        mlflow.log_param("model_type", "LogisticRegression")
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
        mlflow.log_artifact(str(model_path), artifact_path="model")
        mlflow.log_artifact(str(metadata_path), artifact_path="metadata")
    mlflow_status = "logged"
except Exception as exc:
    mlflow_status = f"skipped ({exc})"

{"artifact_dir": str(ARTIFACT_DIR), "mlflow": mlflow_status}

## Section D - Lifecycle Mapping (Notebook -> MLOps)

This toy flow maps to the lifecycle as follows:

1. Data snapshotting corresponds to data versioning.
2. Train/evaluate corresponds to experiment iteration.
3. Artifact save + metadata corresponds to packaging and lineage.
4. Optional MLflow tracking corresponds to experiment observability.

In production, these same stages move to automated pipelines with tests, approvals, staged deployments, and monitoring loops.

## Business Case Studies & Exceptions

### Case 1: Team stuck in notebook-only delivery

- **Problem**: models perform well offline but fail to deploy repeatedly.
- **Fix**: define reproducible run metadata, artifact standards, and CI checks before scaling platform complexity.
- **Impact**: faster model handoff and fewer "works on my machine" incidents.

### Case 2: Mature lifecycle in a fintech scoring platform

- **Pattern**: tracked experiments + registry approvals + staged deploy + drift alerts.
- **Impact**: lower release risk and better auditability.

### Exceptions

- For low-impact internal analytics models, full CT may be unnecessary.
- For high-risk models, manual governance gates remain essential even with automation.

## Interview Questions & Answers

1. **What is MLOps in one sentence?**  
   It is the practice of operationalizing ML systems with automation, reproducibility, monitoring, and governance.

2. **Why track experiments?**  
   To reproduce results, compare alternatives, and preserve lineage for audits and debugging.

3. **What is the difference between CI/CD and CT?**  
   CI/CD automates software build/release; CT automates model retraining and validation cycles.

4. **Why save data snapshots?**  
   Metrics are only comparable if evaluated on known, versioned data.

5. **What is model lineage?**  
   End-to-end trace from data + code + parameters to a deployed model version.

6. **When should retraining be triggered?**  
   On schedule or when drift/performance thresholds indicate meaningful degradation.

7. **What is a model registry used for?**  
   Managing model versions, approval states, and deployment metadata.

8. **What is the notebook-to-prod gap?**  
   The gap between exploratory code and production-grade, tested, deployable pipelines.

9. **How do you reduce deployment risk?**  
   Use staged rollout strategies with measurable promotion and rollback criteria.

10. **What is continuous monitoring in ML?**  
   Ongoing tracking of data drift, model behavior, performance, and service health after deployment.

11. **Why are seeds important?**  
   They improve reproducibility and help isolate true changes from random variation.

12. **What is the first MLOps step for a small team?**  
   Standardize experiment metadata and artifact packaging before building full platform automation.